# Postnatal SGN integration: scVI, HVG10000

Set the paths marked in the code before running this notebook.

Reproduces the selected HVG10000 run in `Integration_postnatal_period.ipynb`: 985 cells, five sample batches, seed 0, CPU and 400 epochs. Batch and label category order are retained explicitly.

The first run assembles counts from the three stagewise H5AD files using the verified common-gene mapping. Later runs load the saved model and `X_scvi`; UMAP is recalculated and checked against the selected run. Use the `scvi_env` kernel.

Outputs: two UMAP PDFs, the full and split DotPlot PDFs, and one model/H5AD pair. No UMAP coordinates are saved.


In [ ]:
#1. =========== Paths and shared settings
# Replace each ROOT placeholder with the required file or directory path before running.
from pathlib import Path
from importlib.metadata import version
import hashlib
import json
import logging
import os
import tempfile
import warnings
from copy import deepcopy

warnings.filterwarnings("ignore", message="IProgress not found.*", module=r"tqdm\..*")
warnings.filterwarnings("ignore", message=".*does not have many workers.*", module=r"lightning\.pytorch\..*")
warnings.filterwarnings("ignore", message=".*LeafSpec.*is deprecated.*", module=r"lightning\.pytorch\..*")

import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
import scvi
import torch
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.lines import Line2D
from matplotlib.ticker import FormatStrFormatter
from matplotlib.transforms import Bbox
from scipy import sparse
from scipy.spatial.distance import cdist

ROOT = Path(".")  # Set the project root before running this template.
# Set the result file path; uses SCVI_10000_Postnatal_Inte_adata.h5ad generated by postnatal scVI integration.
FIGURE_DIR = ROOT
# Set the data file or directory path.
RESULT_DIR = ROOT
RESULT_PREFIX = "SCVI_10000_Postnatal_Inte_"
RANDOM_STATE = 0
N_THREADS = 4
COUNTS_LAYER = "counts"
BATCH_KEY = "integration_batch"
LABELS_KEY = "scanvi_label"
RECIPE = "postnatal_scvi_HVG10000_original_batch_order_v2"
EXPECTED_LATENT_SHA256 = "e761fcb7a1d937b6f7deab5e2baf1a78ca5cecd5e69ffb1766268b9ac2b8ba89"
EXPECTED_UMAP_SHA256 = "ebb2ebf7623385171606fc3a4e7a992de81f0e6ea7048674f67718a16b46de8a"
stages = ["P1", "P7", "P28"]
label_order = ["Type Ia", "Type Ib", "Type Ic", "Type II"]
groups = [f"{s} {t}" for s in stages for t in label_order]
postbirth_stage_colors = {
    "P1 Type Ia": "#E37222", "P1 Type Ib": "#C50084",
    "P1 Type Ic": "#00B2A9", "P1 Type II": "#7D5CC6",
    "P7 Type Ia": "#2A6EBB", "P7 Type Ib": "#69BE28",
    "P7 Type Ic": "#CD202C", "P7 Type II": "#F0AB00",
    "P28 Type Ia": "#7E6148", "P28 Type Ib": "#F39B7F",
    "P28 Type Ic": "#3C5488", "P28 Type II": "#747678",
}

ad.settings.allow_write_nullable_strings = True
scvi.settings.verbosity = logging.WARNING
scvi.settings.dl_num_workers = 0
scvi.settings.dl_persistent_workers = False
for name in ("lightning.pytorch", "pytorch_lightning", "lightning.fabric.utilities.seed"):
    logging.getLogger(name).setLevel(logging.WARNING)
torch.set_num_threads(N_THREADS)
plt.rcParams.update({"pdf.fonttype": 42, "savefig.bbox": None})


def matrix_hash(matrix):
    matrix = matrix.copy().tocsr()
    matrix.sort_indices()
    digest = hashlib.sha256()
    for values, dtype in [(matrix.shape, "<i8"), (matrix.indptr, "<i8"),
                          (matrix.indices, "<i8"), (matrix.data, "<i4")]:
        digest.update(np.asarray(values, dtype=dtype).tobytes())
    return digest.hexdigest()


def array_hash(array):
    return hashlib.sha256(np.ascontiguousarray(array, dtype="<f4").tobytes()).hexdigest()


In [ ]:
#2. =========== Load saved data or assemble stagewise counts
# Set the data file or directory path.
INPUT_DIR = ROOT
# Set the data file path; uses postnatal_prepared.h5ad generated by fig1_Postnatal_preparation.py.
MAPPING_FILE = ROOT

# Set the result file path; uses SCVI_10000_Postnatal_Inte_model.pt generated by postnatal scVI integration.
model_file = ROOT
# Set the result file path; uses SCVI_10000_Postnatal_Inte_adata.h5ad generated by postnatal scVI integration.
data_file = ROOT
if model_file.exists() != data_file.exists():
    raise RuntimeError("The saved scVI pair is incomplete; both model.pt and adata.h5ad are required.")

reference = ad.read_h5ad(MAPPING_FILE)
batch_order = reference.obs[BATCH_KEY].cat.categories.tolist()
reuse_model = False
if model_file.is_file() and data_file.is_file():
    adata_hvg10000 = ad.read_h5ad(data_file)
    reuse_model = (
        adata_hvg10000.uns.get("scvi_recipe") == RECIPE
        and adata_hvg10000.obs[BATCH_KEY].cat.categories.tolist() == batch_order
    )
    if not reuse_model:
        print("Saved result has different batch encoding; retraining the selected recipe.")

if reuse_model:
    prepared = ad.AnnData(
        X=adata_hvg10000.raw.X.copy(), obs=adata_hvg10000.obs.copy(),
        var=adata_hvg10000.raw.var.copy(),
    )
    prepared.layers[COUNTS_LAYER] = prepared.X.copy()
    prepared.uns = deepcopy(adata_hvg10000.uns["preparation_provenance"])
else:
    # Set the stage result path; uses {stage}_cluster_intergration.h5ad generated by P1, P7 or P28 clustering.
    data = {s: ad.read_h5ad(ROOT) for s in stages}
    axis = reference.var.copy()
    assert reference.uns["gene_reference"]["release"] == 115
    aligned = []
    for stage in stages:
        source = data[stage]
        positions = axis[f"{stage}_source_column"].to_numpy(dtype=int)
        assert source.var_names[positions].tolist() == axis[f"{stage}_source_symbol"].astype(str).tolist()
        assert source.var.iloc[positions]["gene_id_original"].astype(str).tolist() == axis[f"{stage}_source_gene_id"].astype(str).tolist()
        assert source.obs["stage_analysis"].astype(str).eq(stage).all()
        aligned.append(source.layers[COUNTS_LAYER][:, positions].copy())

    obs = pd.concat([data[s].obs.copy() for s in stages], axis=0, sort=False).copy()
    # Standardize stage-specific metadata types for H5AD.
    for key in obs.select_dtypes(include="object"):
        present = obs[key].dropna()
        if present.map(lambda x: isinstance(x, (bool, np.bool_))).all():
            obs[key] = obs[key].astype("boolean")
        elif present.map(lambda x: isinstance(x, (int, float, np.number))).all():
            obs[key] = pd.to_numeric(obs[key])
        else:
            obs[key] = obs[key].astype("string")
    assert obs.index.equals(reference.obs_names)
    for key in ("stage_analysis", "cell_type", BATCH_KEY, LABELS_KEY):
        assert obs[key].astype(str).equals(reference.obs[key].astype(str)), f"Changed {key}."
    for key in ("total_counts_analysis", "n_genes_analysis"):
        obs[key.replace("_analysis", "_stagewise")] = obs[key].copy()
    joint_counts = sparse.vstack(aligned, format="csr", dtype=np.int32)
    joint_counts.sort_indices()
    assert (joint_counts != reference.layers[COUNTS_LAYER]).nnz == 0
    obs["total_counts_analysis"] = np.asarray(joint_counts.sum(axis=1)).ravel()
    obs["n_genes_analysis"] = np.asarray((joint_counts > 0).sum(axis=1)).ravel()
    obs["counts_retained_fraction"] = obs["total_counts_analysis"] / obs["total_counts_stagewise"]
    # Keep the original category order, with P7 before P28.
    for key in reference.obs:
        if isinstance(reference.obs[key].dtype, pd.CategoricalDtype):
            obs[key] = pd.Categorical(
                obs[key], categories=reference.obs[key].cat.categories,
                ordered=reference.obs[key].cat.ordered,
            )
    prepared = ad.AnnData(X=joint_counts.copy(), obs=obs, var=axis)
    prepared.layers[COUNTS_LAYER] = joint_counts
    prepared.uns = deepcopy(reference.uns)
    prepared.uns["source_contracts"] = {s: deepcopy(data[s].uns) for s in stages}
    # Set the stage result path; uses {stage}_cluster_intergration.h5ad generated by P1, P7 or P28 clustering.
    prepared.uns["input_file_sha256"] = {
        s: hashlib.sha256((ROOT).read_bytes()).hexdigest()
        for s in stages
    }
    del data, aligned

for key in ("stage_analysis", "stage_standardized", "cell_type", LABELS_KEY, BATCH_KEY, "batch_id_global"):
    pd.testing.assert_series_equal(prepared.obs[key], reference.obs[key])
del reference

assert prepared.shape == (985, 10242)
assert prepared.obs_names.is_unique and prepared.var_names.is_unique
assert prepared.obs[BATCH_KEY].nunique() == 5
assert prepared.obs[LABELS_KEY].astype(str).equals(prepared.obs["cell_type"].astype(str))
assert not prepared.var[["mt_feature", "hemoglobin_feature", "technical_feature"]].any().any()
assert matrix_hash(prepared.layers[COUNTS_LAYER]) == "e9f1a6813fced9243d04684359ef1f04a9b96501034e6632f37d079b31137c95"

meta = prepared.obs[["stage_analysis", "cell_type", BATCH_KEY]].copy()
meta["stage_celltype"] = pd.Categorical(
    meta["stage_analysis"].astype(str) + " " + meta["cell_type"].astype(str),
    categories=groups, ordered=True,
)
assert meta.notna().all().all()
codes = meta["stage_celltype"].cat.codes.to_numpy()
counts = np.bincount(codes, minlength=len(groups))
assert counts.tolist() == [145, 114, 37, 18, 160, 121, 55, 51, 52, 93, 117, 22]
print(f"Input: {prepared.n_obs} cells, {prepared.n_vars} common genes, 5 batches.")


In [ ]:
#3. =========== Full-gene normalization and pooled HVG ranking
HVG_N = 10000
HVG_FLAVOR = "seurat"
HVG_N_BINS = 20
HVG_RANK_REFERENCE_N = 5000  # Original pooled dispersion calculation
TARGET_SUM = 1e4

hvg_input = ad.AnnData(
    X=prepared.layers[COUNTS_LAYER].astype(np.float64),
    obs=prepared.obs.copy(), var=prepared.var.copy(),
)
sc.pp.normalize_total(hvg_input, target_sum=TARGET_SUM)
sc.pp.log1p(hvg_input)
hvg_stats = sc.pp.highly_variable_genes(
    hvg_input, flavor=HVG_FLAVOR, n_top_genes=HVG_RANK_REFERENCE_N,
    n_bins=HVG_N_BINS, batch_key=None, inplace=False,
)
hvg_rank = hvg_stats.loc[np.isfinite(hvg_stats["dispersions_norm"])]
hvg_rank = hvg_rank.sort_index().sort_values("dispersions_norm", ascending=False, kind="stable")
assert len(hvg_rank) >= HVG_N
keep_10000 = prepared.var_names.isin(hvg_rank.index[:HVG_N])

if reuse_model:
    assert adata_hvg10000.var_names.equals(prepared.var_names[keep_10000])
else:
    adata_hvg10000 = ad.AnnData(
        X=prepared.layers[COUNTS_LAYER][:, keep_10000].copy(),
        obs=prepared.obs.copy(), var=prepared.var.loc[keep_10000].copy(),
    )
    adata_hvg10000.layers[COUNTS_LAYER] = adata_hvg10000.X.copy()
    adata_hvg10000.uns["preparation_provenance"] = deepcopy(prepared.uns)
    adata_hvg10000.var["highly_variable"] = True
    adata_hvg10000.var["dispersions_norm"] = hvg_stats.loc[adata_hvg10000.var_names, "dispersions_norm"]
    for key in ("total_counts_analysis", "n_genes_analysis"):
        adata_hvg10000.obs[key + "_pre_HVG"] = adata_hvg10000.obs[key].copy()
    adata_hvg10000.obs["total_counts_analysis"] = np.asarray(adata_hvg10000.X.sum(axis=1)).ravel()
    adata_hvg10000.obs["n_genes_analysis"] = np.asarray((adata_hvg10000.X > 0).sum(axis=1)).ravel()
    adata_hvg10000.obs["counts_retained_fraction"] = (
        adata_hvg10000.obs["total_counts_analysis"] / adata_hvg10000.obs["total_counts_stagewise"]
    )
assert adata_hvg10000.shape == (985, HVG_N)
assert (adata_hvg10000.layers[COUNTS_LAYER] != prepared.layers[COUNTS_LAYER][:, keep_10000]).nnz == 0


In [ ]:
#4. =========== Train or reload scVI
SCVI_MAX_EPOCHS = 400
N_LATENT = 30
N_LAYERS = 1
N_HIDDEN = 128
DROPOUT_RATE = 0.1
GENE_LIKELIHOOD = "zinb"
BATCH_SIZE = 128
TRAIN_SIZE = 0.9
LEARNING_RATE = 1e-3
KL_WARMUP_EPOCHS = 400
ACCELERATOR = "cpu"  # Matches the selected result

settings = {
    "hvg": HVG_N, "hvg_flavor": HVG_FLAVOR, "hvg_n_bins": HVG_N_BINS,
    "hvg_rank_reference_n": HVG_RANK_REFERENCE_N,
    "batch_categories": batch_order, "label_categories": label_order,
    "target_sum": TARGET_SUM, "seed": RANDOM_STATE, "threads": N_THREADS,
    "batch_key": BATCH_KEY, "labels_key": LABELS_KEY, "epochs": SCVI_MAX_EPOCHS,
    "n_latent": N_LATENT, "n_layers": N_LAYERS, "n_hidden": N_HIDDEN,
    "dropout_rate": DROPOUT_RATE, "gene_likelihood": GENE_LIKELIHOOD,
    "dispersion": "gene", "batch_size": BATCH_SIZE, "train_size": TRAIN_SIZE,
    "lr": LEARNING_RATE, "kl_warmup": KL_WARMUP_EPOCHS, "accelerator": ACCELERATOR,
}
scvi.settings.seed = RANDOM_STATE
before_hash = matrix_hash(adata_hvg10000.layers[COUNTS_LAYER])

if reuse_model:
    assert json.loads(adata_hvg10000.uns["scvi_settings"]) == settings, "Settings differ from the saved run."
    scvi_hvg10000 = scvi.model.SCVI.load(
        str(RESULT_DIR), adata=adata_hvg10000, prefix=RESULT_PREFIX,
        accelerator=ACCELERATOR, device=1,
    )
    print("scVI: saved model and X_scvi loaded; training skipped.")
else:
    scvi.model.SCVI.setup_anndata(
        adata_hvg10000, layer=COUNTS_LAYER, batch_key=BATCH_KEY, labels_key=LABELS_KEY,
    )
    scvi_hvg10000 = scvi.model.SCVI(
        adata_hvg10000, n_latent=N_LATENT, n_layers=N_LAYERS, n_hidden=N_HIDDEN,
        dropout_rate=DROPOUT_RATE, gene_likelihood=GENE_LIKELIHOOD, dispersion="gene",
    )
    scvi_hvg10000.train(
        max_epochs=SCVI_MAX_EPOCHS, accelerator=ACCELERATOR, devices=1,
        batch_size=BATCH_SIZE, train_size=TRAIN_SIZE, validation_size=1 - TRAIN_SIZE,
        shuffle_set_split=True, early_stopping=False,
        plan_kwargs={"lr": LEARNING_RATE, "n_epochs_kl_warmup": KL_WARMUP_EPOCHS},
        check_val_every_n_epoch=1, enable_progress_bar=False,
        enable_model_summary=False, enable_checkpointing=False,
        logger=None, log_save_dir=None, log_every_n_steps=1, benchmark=False,
    )
    scvi_history_hvg10000 = pd.concat([
        scvi_hvg10000.history[key].iloc[:, 0].rename(label)
        for key, label in [("elbo_train", "train"), ("elbo_validation", "validation")]
    ], axis=1).astype(float)
    assert scvi_hvg10000.trainer.current_epoch == SCVI_MAX_EPOCHS
    assert len(scvi_history_hvg10000) == SCVI_MAX_EPOCHS
    assert np.isfinite(scvi_history_hvg10000.to_numpy()).all()
    adata_hvg10000.obsm["X_scvi"] = scvi_hvg10000.get_latent_representation()

latent = np.asarray(adata_hvg10000.obsm["X_scvi"])
assert latent.shape == (985, N_LATENT) and np.isfinite(latent).all()
assert matrix_hash(adata_hvg10000.layers[COUNTS_LAYER]) == before_hash
assert scvi_hvg10000.is_trained

assert scvi_hvg10000.adata_manager.get_state_registry("batch").categorical_mapping.tolist() == batch_order
assert scvi_hvg10000.adata_manager.get_state_registry("labels").categorical_mapping.tolist() == label_order
assert array_hash(latent) == EXPECTED_LATENT_SHA256, "X_scvi differs from the selected original run."


In [ ]:
#5. =========== Neighbors and UMAP
N_NEIGHBORS = 30
UMAP_MIN_DIST = 0.35
UMAP_SPREAD = 1.0
UMAP_EPOCHS = 500

view_hvg10000 = ad.AnnData(obs=meta.copy())
view_hvg10000.obsm["X_latent"] = latent.copy()
sc.pp.neighbors(
    view_hvg10000, use_rep="X_latent", n_neighbors=N_NEIGHBORS,
    metric="euclidean", transformer="sklearn", random_state=RANDOM_STATE,
)
sc.tl.umap(
    view_hvg10000, min_dist=UMAP_MIN_DIST, spread=UMAP_SPREAD,
    maxiter=UMAP_EPOCHS, random_state=RANDOM_STATE,
)

assert array_hash(view_hvg10000.obsm["X_umap"]) == EXPECTED_UMAP_SHA256, "UMAP differs from the selected original run."


In [ ]:
#6. =========== Save the verified model and data
if not reuse_model:
    adata_hvg10000.uns["scvi_recipe"] = RECIPE
    adata_hvg10000.uns["reproduction_fingerprints"] = {
        "X_scvi_sha256": array_hash(latent),
        "X_umap_sha256": array_hash(view_hvg10000.obsm["X_umap"]),
        "encoding": "SHA256 of C-order little-endian float32; UMAP coordinates are not saved.",
    }
    adata_hvg10000.obs["stage_celltype"] = meta["stage_celltype"].copy()
    adata_hvg10000.uns["stage_celltype_colors"] = [postbirth_stage_colors[g] for g in groups]
    # Retain all common-gene counts for later expression plots.
    adata_hvg10000.raw = prepared
    adata_hvg10000.uns["raw_definition"] = "Unnormalized integer UMI counts for all 10242 common genes."
    adata_hvg10000.uns["scvi_settings"] = json.dumps(settings, sort_keys=True)
    adata_hvg10000.uns["software_versions"] = {
        name: version(name) for name in ("scvi-tools", "scanpy", "anndata", "torch", "numpy", "umap-learn")
    }
    scvi_history_hvg10000.index = scvi_history_hvg10000.index.astype(str)
    adata_hvg10000.uns["training_history"] = scvi_history_hvg10000
    RESULT_DIR.mkdir(parents=True, exist_ok=True)
    # Set the result file path; uses SCVI_10000_Postnatal_Inte_adata.h5ad generated by postnatal scVI integration.
    temporary_root = ROOT
    temporary_root.mkdir(parents=True, exist_ok=True)
    try:
        with tempfile.TemporaryDirectory(dir=temporary_root) as work:
            scvi_hvg10000.save(
                work, prefix=RESULT_PREFIX, overwrite=True,
                save_anndata=True, compression="gzip",
            )
            checked = ad.read_h5ad(Path(work) / data_file.name)
            assert checked.obs_names.equals(adata_hvg10000.obs_names)
            assert checked.var_names.equals(adata_hvg10000.var_names)
            assert matrix_hash(checked.layers[COUNTS_LAYER]) == before_hash
            assert matrix_hash(checked.raw.X) == matrix_hash(prepared.layers[COUNTS_LAYER])
            pd.testing.assert_frame_equal(
                checked.obs, adata_hvg10000.obs, check_dtype=False, check_categorical=False,
            )
            for key in (BATCH_KEY, LABELS_KEY, "stage_analysis", "stage_celltype"):
                assert checked.obs[key].cat.categories.tolist() == adata_hvg10000.obs[key].cat.categories.tolist()
                assert checked.obs[key].cat.ordered == adata_hvg10000.obs[key].cat.ordered
            assert checked.obs[BATCH_KEY].cat.categories.tolist() == batch_order
            assert "X_umap" not in checked.obsm
            np.testing.assert_array_equal(checked.obsm["X_scvi"], latent)
            reloaded = scvi.model.SCVI.load(
                work, adata=checked, prefix=RESULT_PREFIX, accelerator=ACCELERATOR, device=1,
            )
            np.testing.assert_array_equal(reloaded.get_latent_representation(), latent)
            del checked, reloaded
            for destination in (data_file, model_file):
                os.replace(Path(work) / destination.name, destination)
    finally:
        if temporary_root.exists() and not any(temporary_root.iterdir()):
            temporary_root.rmdir()
    print(f"scVI: {SCVI_MAX_EPOCHS} epochs; model/data read-back verified and saved.")


In [ ]:
#7. =========== Final UMAPs
POINT_SIZE = 2.70          # Scatter area, pt^2
POINT_ALPHA = 0.82
LABEL_FONTSIZE = 6.75
LEGEND_FONTSIZE = 7.5
AXIS_SIZE = 3.8            # Square axes, inches
FIGURE_DPI = 110           # Notebook display; PDFs remain vector
LINE_DISTANCE_PX = 12
LABEL_SEARCH_RADII = (0, 12, 24, 36, 48)

FIGURE_DIR.mkdir(parents=True, exist_ok=True)
draw_order = np.random.default_rng(RANDOM_STATE).permutation(len(meta))
point_colors = np.array([postbirth_stage_colors[g] for g in meta["stage_celltype"].astype(str)])


def draw_umap_pair(view, title):
    xy = np.asarray(view.obsm["X_umap"])
    assert xy.shape == (len(meta), 2) and np.isfinite(xy).all()

    lower, upper = xy.min(axis=0), xy.max(axis=0)
    pad = np.maximum((upper - lower) * 0.08, 0.1)
    limits = (lower - pad, upper + pad)

    for version in ("legend", "labels"):
        height = AXIS_SIZE + 1.1
        width = AXIS_SIZE + (2.7 if version == "legend" else 1.1)

        fig = plt.figure(
            figsize=(width, height),
            dpi=FIGURE_DPI,
            facecolor="white",
        )
        ax = fig.add_axes([
            0.65 / width, 0.6 / height,
            AXIS_SIZE / width, AXIS_SIZE / height,
        ])

        ax.scatter(
            xy[draw_order, 0],
            xy[draw_order, 1],
            c=point_colors[draw_order],
            s=POINT_SIZE,
            alpha=POINT_ALPHA,
            linewidths=0,
        )
        ax.set(
            xlim=(limits[0][0], limits[1][0]),
            ylim=(limits[0][1], limits[1][1]),
            xlabel="UMAP1",
            ylabel="UMAP2",
            title=f"{title} | {version}",
        )
        ax.set_box_aspect(1)
        ax.tick_params(labelsize=8)
        ax.spines[["top", "right"]].set_visible(False)

        for side in ("left", "bottom"):
            ax.spines[side].set_linewidth(1.15)

        if version == "legend":
            handles = [
                Line2D(
                    [], [],
                    marker="o",
                    linestyle="none",
                    color=postbirth_stage_colors[group],
                    markersize=5,
                    label=f"{group.replace('_', ' ')} ({counts[i]})",
                )
                for i, group in enumerate(groups)
            ]
            ax.legend(
                handles=handles,
                loc="center left",
                bbox_to_anchor=(1.02, 0.5),
                frameon=False,
                fontsize=LEGEND_FONTSIZE,
                handletextpad=0.35,
            )

        else:
            anchors, texts = [], []

            for i, group in enumerate(groups):
                points = xy[codes == i]
                anchor = points[np.argmin(cdist(points, points).sum(axis=1))]
                anchors.append(anchor)

                texts.append(ax.text(
                    *anchor,
                    group.replace("_", " "),
                    ha="center",
                    va="center",
                    fontsize=LABEL_FONTSIZE,
                    color="black",
                    path_effects=[
                        pe.withStroke(linewidth=1.45, foreground="white")
                    ],
                    zorder=4,
                ))

            fig.canvas.draw()
            renderer = fig.canvas.get_renderer()
            axis_box = ax.get_window_extent(renderer)
            pixels = ax.transData.transform(xy)
            placed = []
            angles = np.linspace(0, 2 * np.pi, 16, endpoint=False)

            for anchor, text in zip(anchors, texts):
                center = ax.transData.transform(anchor)
                box = text.get_window_extent(renderer)
                w, h = box.width + 4, box.height + 3
                choices = []

                for radius in LABEL_SEARCH_RADII:
                    for angle in ([0] if radius == 0 else angles):
                        pos = center + radius * np.array([
                            np.cos(angle), np.sin(angle)
                        ])
                        candidate = Bbox.from_bounds(
                            pos[0] - w / 2, pos[1] - h / 2, w, h
                        )
                        inside = (
                            axis_box.contains(*candidate.p0)
                            and axis_box.contains(*candidate.p1)
                        )
                        if not inside:
                            continue

                        overlap = sum(
                            candidate.overlaps(other) for other in placed
                        )
                        covered = np.sum(
                            (pixels[:, 0] >= candidate.x0)
                            & (pixels[:, 0] <= candidate.x1)
                            & (pixels[:, 1] >= candidate.y0)
                            & (pixels[:, 1] <= candidate.y1)
                        )

                        if radius == 0 and overlap == 0 and covered <= 8:
                            choices = [(0, pos, candidate)]
                            break

                        cost = 1e6 * overlap + 20 * covered + radius
                        choices.append((cost, pos, candidate))

                    if choices and choices[0][0] == 0:
                        break

                assert choices, "Increase the axes size to fit labels."
                _, pos, candidate = min(choices, key=lambda item: item[0])
                text.set_position(ax.transData.inverted().transform(pos))
                placed.append(candidate)

                vector = center - pos
                if np.linalg.norm(vector) > 0:
                    scale = min(
                        w / (2 * max(abs(vector[0]), 1e-9)),
                        h / (2 * max(abs(vector[1]), 1e-9)),
                    )
                    edge = pos + min(scale, 1) * vector

                    if np.linalg.norm(center - edge) > LINE_DISTANCE_PX:
                        segment = ax.transData.inverted().transform([
                            edge, center
                        ])
                        ax.plot(
                            segment[:, 0], segment[:, 1],
                            color="0.32", lw=0.42, alpha=0.70, zorder=3,
                        )

            fig.canvas.draw()
            boxes = [
                t.get_window_extent(fig.canvas.get_renderer())
                for t in texts
            ]
            if any(
                boxes[i].overlaps(boxes[j])
                for i in range(len(boxes))
                for j in range(i + 1, len(boxes))
            ):
                warnings.warn(
                    f"{title}: some labels overlap; inspect the plot."
                )

        fig.canvas.draw()
        extent = ax.get_window_extent(fig.canvas.get_renderer())
        assert abs(extent.width - extent.height) < 1

        name = "UMAP_legend" if version == "legend" else "UMAP_label"
        # Set the result file path; uses SCVI_10000_Postnatal_Inte_adata.h5ad generated by postnatal scVI integration.
        fig.savefig(ROOT, facecolor="white")
        plt.show()
        plt.close(fig)


draw_umap_pair(view_hvg10000, "HVG10000 | scVI")


In [ ]:
#8. =========== Marker expression in all common genes
MARKER_GROUPS = {
    "Type Ia": ["Calb2", "Pcdh20", "Rxrg", "Trim54", "B3gat1", "Cacna1b"],
    "Type Ib": ["Calb1", "Ttn", "Kcnd2", "Pls3", "Plcb4", "Kcnh2", "Edil3"],
    "Type Ic": ["Lypd1", "Grm8", "Kcnc2", "Sphkap", "Cd164l2", "Rgs12", "Pcsk2"],
    "Type II": ["Ngfr", "Plk5", "Th", "Calca", "Cux2", "Npy", "Cilp", "Grik3", "Prph"],
    "Ib/Ic shared": ["Ntng1", "Nalcn", "Ncald"],
    "Development": ["Cxcl14", "Etv4", "Gabrg2", "Gata3"],
    "Neuron/SGN": ["Snap25", "Stmn2", "Elavl3", "Rbfox3", "Isl1", "Mafb", "Nefm", "Nefh", "Tubb3", "Nefl"],
    "Glia": ["Sox10", "Mpz", "Plp1", "Mbp", "Pmp22"],
    "Immune": ["Ptprc", "Lyz2", "Tyrobp", "C1qa", "Csf1r", "Ctss"],
    "Erythroid": ["Gypa", "Alas2", "Slc4a1"],
    "Epithelium/hair": ["Epcam", "Krt8", "Krt18", "Myo7a", "Pou4f3"],
    "Mesenchyme/vessel": ["Col1a1", "Col1a2", "Dcn", "Pecam1", "Kdr"],
}
MARKER_COLORS = {
    "Type Ia": "#B7864A", "Type Ib": "#B65F64", "Type Ic": "#557B9F",
    "Type II": "#519A94", "Ib/Ic shared": "#929E50", "Development": "#9A6E9C",
    "Neuron/SGN": "#6F748C", "Glia": "#68A5BA", "Immune": "#B22222",
    "Erythroid": "#C0839E", "Epithelium/hair": "#7A3CB7", "Mesenchyme/vessel": "#6E9368",
}
PANELS = (
    ("SGN subtype and developmental markers", tuple(MARKER_GROUPS)[:6]),
    ("Neuronal identity and exclusion checks", tuple(MARKER_GROUPS)[6:]),
)

marker_genes = [g for panel in MARKER_GROUPS.values() for g in panel]
assert len(marker_genes) == len(set(marker_genes))
symbols = prepared.var["gene_symbol"].astype(str).to_numpy()
positions = [np.flatnonzero(symbols == g) for g in marker_genes]
assert all(len(p) <= 1 for p in positions), "Resolve duplicated gene symbols before plotting."
available = np.array([len(p) == 1 for p in positions])
assert available.any(), "No requested markers are available."
gene_positions = [int(p[0]) for p in positions if len(p) == 1]

expr = hvg_input.X[:, gene_positions]
expr = expr.toarray() if sparse.issparse(expr) else np.asarray(expr)
marker_counts = prepared.layers["counts"][:, gene_positions]
marker_counts = marker_counts.toarray() if sparse.issparse(marker_counts) else np.asarray(marker_counts)
assert np.isfinite(expr).all() and np.isfinite(marker_counts).all()

row_pairs = [(t, s) for t in label_order for s in stages]
row_keys = [f"{s} {t}" for t, s in row_pairs]
row_masks = [meta["stage_celltype"].astype(str).eq(g).to_numpy() for g in row_keys]
row_sizes = np.array([mask.sum() for mask in row_masks])
assert (row_sizes > 0).all() and row_sizes.sum() == prepared.n_obs

mean_matrix = np.full((len(row_keys), len(marker_genes)), np.nan)
fraction_matrix = np.full_like(mean_matrix, np.nan)
for i, mask in enumerate(row_masks):
    mean_matrix[i, available] = expr[mask].mean(axis=0)
    fraction_matrix[i, available] = (marker_counts[mask] > 0).mean(axis=0)

values = mean_matrix[:, available]
sd = values.std(axis=0)
scaled_matrix = mean_matrix.copy()
scaled_matrix[:, available] = np.divide(
    values - values.mean(axis=0), sd, out=np.zeros_like(values), where=sd > 0
)
postbirth_marker_stats = {
    "mean_log1p": pd.DataFrame(mean_matrix, index=row_keys, columns=marker_genes),
    "fraction": pd.DataFrame(fraction_matrix, index=row_keys, columns=marker_genes),
    "scaled_mean": pd.DataFrame(scaled_matrix, index=row_keys, columns=marker_genes),
    "n_cells": pd.Series(row_sizes, index=row_keys),
}
postbirth_marker_coverage = pd.DataFrame([
    {"group": group, "gene": g, "available_in_shared_input": bool(available[marker_genes.index(g)])}
    for group, panel in MARKER_GROUPS.items() for g in panel
])
missing = [g for g, found in zip(marker_genes, available) if not found]
if missing:
    print("Unavailable in the shared input: " + ", ".join(missing))


In [ ]:
#9. =========== Full and split DotPlots
MAX_DOT_AREA = 144
COLUMN_GAP_RATIO = 0.5
ROW_STEP_IN = 0.25
GENE_LABEL_ROTATION = 55
GROUP_LABEL_ROTATION = 60
DOT_FONT_SIZE = 8

cmap = LinearSegmentedColormap.from_list(
    "marker_purple", ["#F1F1F1", "#E8C6DE", "#C67ABA", "#952E7D", "#69005F"]
)
color_norm = Normalize(-1, 1, clip=True)
global_max_area = MAX_DOT_AREA * np.nanmax(fraction_matrix)
assert global_max_area > 0, "All available markers have zero expression."
diameter_in = np.sqrt(global_max_area) / 72
column_step_in = (1 + COLUMN_GAP_RATIO) * diameter_in
row_step_in = max(ROW_STEP_IN, 1.5 * diameter_in)
row_labels = [f"{t} | {s} ({n})" for (t, s), n in zip(row_pairs, row_sizes)]

all_panels = [("Postnatal SGN markers | scVI HVG10000", tuple(MARKER_GROUPS)), *PANELS]
for panel_number, (title, panel_groups) in enumerate(all_panels):
    panel_genes = [g for group in panel_groups for g in MARKER_GROUPS[group]]
    columns = [marker_genes.index(g) for g in panel_genes]
    pct, z = fraction_matrix[:, columns], scaled_matrix[:, columns]
    nr, nc = pct.shape
    left, right, bottom, top = 1.9, 0.35, 1.15, 2.3
    matrix_width, matrix_height = nc * column_step_in, nr * row_step_in
    width, height = max(left + matrix_width + right, 6.2), bottom + matrix_height + top
    fig = plt.figure(figsize=(width, height), dpi=FIGURE_DPI, facecolor="white")
    ax = fig.add_axes([left / width, bottom / height, matrix_width / width, matrix_height / height])

    x, y = np.meshgrid(np.arange(nc), np.arange(nr))
    detected = np.isfinite(pct) & (pct > 0)
    dots = ax.scatter(
        x[detected], y[detected], s=MAX_DOT_AREA * pct[detected], c=z[detected],
        cmap=cmap, norm=color_norm, edgecolors="none", linewidths=0,
    )
    unavailable = ~np.isfinite(pct)
    ax.scatter(x[unavailable], y[unavailable], s=8, marker="x", color="#BBBBBB", linewidths=0.6)
    ax.set(xlim=(-0.5, nc - 0.5), ylim=(nr - 0.5, -0.5))
    ax.set_xticks(np.arange(nc), panel_genes)
    ax.set_yticks(np.arange(nr), row_labels)
    ax.tick_params(axis="both", length=0, labelsize=DOT_FONT_SIZE)
    plt.setp(ax.get_xticklabels(), rotation=GENE_LABEL_ROTATION, ha="right", rotation_mode="anchor")
    ax.spines[:].set_visible(False)
    ax.grid(False)

    offset, headings = 0, []
    for group in panel_groups:
        length = len(MARKER_GROUPS[group])
        ax.plot(
            [offset - 0.35, offset + length - 0.65], [1.015, 1.015],
            transform=ax.get_xaxis_transform(), color=MARKER_COLORS[group],
            lw=3, solid_capstyle="butt", clip_on=False,
        )
        headings.append(ax.text(
            offset + (length - 1) / 2, 1.04, group, transform=ax.get_xaxis_transform(),
            ha="center", va="bottom", rotation=GROUP_LABEL_ROTATION, fontsize=DOT_FONT_SIZE,
        ))
        offset += length

    legend_y = bottom + matrix_height + 1.5
    cax = fig.add_axes([left / width, legend_y / height, 1.5 / width, 0.11 / height])
    cb = fig.colorbar(
        dots, cax=cax, orientation="horizontal", ticks=[-1, -0.5, 0, 0.5, 1],
        format=FormatStrFormatter("%.1f"),
    )
    cb.outline.set_visible(False)
    cb.ax.tick_params(labelsize=DOT_FONT_SIZE - 1, length=2)
    cax.set_title("Avg. Exp. (gene-wise z-score)", fontsize=DOT_FONT_SIZE, pad=5)

    size_ax = fig.add_axes([(left + 2.0) / width, (legend_y - 0.13) / height, 2.1 / width, 0.4 / height])
    percentages = np.array([0, 25, 50, 75, 100])
    size_ax.scatter(
        np.arange(5), np.repeat(0.55, 5), s=MAX_DOT_AREA * percentages / 100,
        color="#777777", edgecolors="none",
    )
    for i, value in enumerate(percentages):
        size_ax.text(i, -0.03, str(value), ha="center", va="top", fontsize=DOT_FONT_SIZE)
    size_ax.set(xlim=(-0.55, 4.55), ylim=(-0.45, 1.1), title="% Exp.")
    size_ax.title.set_fontsize(DOT_FONT_SIZE)
    size_ax.axis("off")
    if unavailable.any():
        fig.text(
            left / width, 0.1 / height, "x: unavailable in the shared input",
            fontsize=DOT_FONT_SIZE, color="#777777",
        )
    fig.suptitle(title, y=1 - 0.12 / height, fontsize=10)

    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    spacing_px = ax.transData.transform((1, 0))[0] - ax.transData.transform((0, 0))[0]
    dmax_px = np.sqrt(global_max_area) * fig.dpi / 72
    assert np.isclose((spacing_px - dmax_px) / dmax_px, COLUMN_GAP_RATIO, atol=0.02)

    text_boxes = [
        t.get_window_extent(renderer)
        for t in [*ax.get_xticklabels(), *ax.get_yticklabels(), *headings]
    ]
    assert all(
        b.x0 >= 0 and b.y0 >= 0 and b.x1 <= fig.bbox.width and b.y1 <= fig.bbox.height
        for b in text_boxes
    ), "Text is clipped."
    header_boxes = [t.get_window_extent(renderer) for t in headings]
    assert not any(
        header_boxes[i].overlaps(header_boxes[j])
        for i in range(len(header_boxes)) for j in range(i + 1, len(header_boxes))
    ), "Marker headings overlap."
    legend_bottom = min(cb.ax.get_tightbbox(renderer).y0, size_ax.get_tightbbox(renderer).y0)
    assert max(b.y1 for b in header_boxes) < legend_bottom, "Marker headings overlap the legends."

    suffix = "full" if panel_number == 0 else str(panel_number)
    # Set the result file path; uses SCVI_10000_Postnatal_Inte_adata.h5ad generated by postnatal scVI integration.
    fig.savefig(ROOT, facecolor="white")
    plt.show()
    plt.close(fig)


# modify figures

In [ ]:
#1. =========== Final UMAP settings
from pathlib import Path
import ast
import hashlib
import io
import json
import logging
import os
import warnings

ROOT = Path(".")  # Set the project root before running this template.
# Set the data file or directory path.
RESULT_DIR = ROOT
RESULT_PREFIX = "SCVI_10000_Postnatal_Inte_"
# Set the color notebook path; uses fig1_Whole_integration.ipynb from whole-period integration.
PALETTE_NOTEBOOK = ROOT
# Set the output directory path.
OUTPUT_DIR = ROOT

RANDOM_STATE = 0
N_THREADS = 4
N_NEIGHBORS = 30
UMAP_MIN_DIST = 0.35
UMAP_SPREAD = 1.0
UMAP_EPOCHS = 500

AXIS_SIZE = 5.6                 # Main plotting area, inches
FIGURE_MARGINS = (0.65, 1.35, 0.60, 0.50)  # Left, right, bottom, top
FIGURE_DPI = 144                # Notebook display only
POINT_SIZE = 3.35               # Scatter area, pt^2
POINT_ALPHA = 0.82
TITLE_FONTSIZE = 10
TITLE_PAD = 10
AXIS_FONTSIZE = 9
AXIS_LABEL_PAD = 5
TICK_FONTSIZE = 7
TICK_PAD = 3
TICK_LENGTH = 3
AXIS_LINEWIDTH = 1.1
LABEL_FONTSIZE = 6.75
LEGEND_FONTSIZE = 7.5
LEGEND_MARKER_SIZE = 5.0         # Marker diameter, pt
LABEL_SEARCH_RADII = (0, 6, 12, 18, 24)  # Local label offsets, pt
LABEL_OFFSETS = {}              # Optional {full category: (dx, dy)}, pt
STAGE_ORDER = ["P1", "P7", "P28"]

os.environ.setdefault("LOKY_MAX_CPU_COUNT", str(N_THREADS))
warnings.filterwarnings("ignore", message="IProgress not found.*")
warnings.filterwarnings("ignore", message=".*LeafSpec.*is deprecated.*")

import anndata as ad
import numpy as np
import scanpy as sc
import scvi
import torch
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.backend_bases import RendererBase
from matplotlib.backends.backend_pdf import PdfFile, RendererPdf, GraphicsContextPdf, Op
from matplotlib.font_manager import FontProperties, findfont
from matplotlib.transforms import Bbox
from scipy.spatial.distance import cdist

findfont(FontProperties(family="Arial"), fallback_to_default=False)
mpl.rcParams.update({
    "font.family": "Arial", "pdf.fonttype": 42,
    "pdf.use14corefonts": False, "text.usetex": False,
})
scvi.settings.verbosity = logging.WARNING
logging.getLogger("lightning.fabric.utilities.seed").setLevel(logging.WARNING)
torch.set_num_threads(N_THREADS)
scvi.settings.seed = RANDOM_STATE


#2. =========== Read the shared literal palettes
def read_shared_colors(path):
    names = {"CELL_ANNOTATION_COLORS", "DEVELOPMENTAL_STAGE_COLORS"}
    palettes = {}
    notebook = json.loads(path.read_text(encoding="utf-8"))
    for cell in notebook["cells"]:
        source = "".join(cell.get("source", []))
        if cell["cell_type"] != "code" or not any(name in source for name in names):
            continue
        source = "\n".join(line for line in source.splitlines()
                           if not line.lstrip().startswith(("%", "!")))
        for node in ast.parse(source).body:
            targets = (node.targets if isinstance(node, ast.Assign) else
                       [node.target] if isinstance(node, ast.AnnAssign) else [])
            for target in targets:
                if isinstance(target, ast.Name) and target.id in names:
                    assert target.id not in palettes, f"Duplicate palette: {target.id}"
                    assert isinstance(node.value, ast.Dict), f"Use a literal dict: {target.id}"
                    palettes[target.id] = ast.literal_eval(node.value)
    assert names == palettes.keys(), "Save both literal palettes in Whole_integration.ipynb."
    for palette in palettes.values():
        assert isinstance(palette, dict) and palette
        assert all(isinstance(k, str) and isinstance(v, str) for k, v in palette.items())
        for color in palette.values():
            mpl.colors.to_rgba(color)
    return palettes["CELL_ANNOTATION_COLORS"], palettes["DEVELOPMENTAL_STAGE_COLORS"]

CELL_ANNOTATION_COLORS, DEVELOPMENTAL_STAGE_COLORS = read_shared_colors(PALETTE_NOTEBOOK)


#3. =========== Load and verify the selected scVI result
def array_hash(values):
    return hashlib.sha256(np.ascontiguousarray(values, dtype="<f4").tobytes()).hexdigest()

# Set the result file path; uses SCVI_10000_Postnatal_Inte_adata.h5ad generated by postnatal scVI integration.
adata_hvg10000 = ad.read_h5ad(ROOT)
assert adata_hvg10000.shape == (985, 10000)
assert adata_hvg10000.obs_names.is_unique
settings = json.loads(adata_hvg10000.uns["scvi_settings"])
batch_order = adata_hvg10000.obs["integration_batch"].cat.categories.tolist()
assert batch_order == settings["batch_categories"]

scvi_hvg10000 = scvi.model.SCVI.load(
    str(RESULT_DIR), adata=adata_hvg10000, prefix=RESULT_PREFIX,
    accelerator="cpu", device=1,
)
assert scvi_hvg10000.is_trained
assert scvi_hvg10000.adata_manager.get_state_registry("batch").categorical_mapping.tolist() == batch_order
assert scvi_hvg10000.adata_manager.get_state_registry("labels").categorical_mapping.tolist() == settings["label_categories"]
np.testing.assert_array_equal(adata_hvg10000.obs["_scvi_batch"],
                              adata_hvg10000.obs["integration_batch"].cat.codes)

latent = np.asarray(adata_hvg10000.obsm["X_scvi"])
fingerprints = adata_hvg10000.uns["reproduction_fingerprints"]
assert latent.shape == (985, 30) and np.isfinite(latent).all()
assert array_hash(latent) == fingerprints["X_scvi_sha256"]
np.testing.assert_array_equal(scvi_hvg10000.get_latent_representation(), latent)

meta = adata_hvg10000.obs[
    ["stage_analysis", "cell_type", "integration_batch", "stage_celltype"]
].copy()
assert not meta.isna().any().any()
groups = meta["stage_celltype"].cat.categories.tolist()
codes = meta["stage_celltype"].cat.codes.to_numpy()
assert len(groups) == 12 and set(codes) == set(range(len(groups)))
annotation_color_keys = {
    group: f"{stage} | {annotation}"
    for stage, annotation, group in meta[
        ["stage_analysis", "cell_type", "stage_celltype"]
    ].drop_duplicates().itertuples(index=False, name=None)
}
assert len(annotation_color_keys) == len(groups)
assert len(set(annotation_color_keys.values())) == len(groups)
assert set(annotation_color_keys.values()) <= CELL_ANNOTATION_COLORS.keys(), "Unmapped cell annotation."
assert set(meta["stage_analysis"].astype(str)) == set(STAGE_ORDER)
assert set(STAGE_ORDER) <= DEVELOPMENTAL_STAGE_COLORS.keys(), "Unmapped stage."
postbirth_stage_colors = {group: CELL_ANNOTATION_COLORS[annotation_color_keys[group]] for group in groups}
STAGE_COLORS = {stage: DEVELOPMENTAL_STAGE_COLORS[stage] for stage in STAGE_ORDER}


#4. =========== Restore and fingerprint the original UMAP
view_hvg10000 = ad.AnnData(obs=meta.copy())
view_hvg10000.obsm["X_latent"] = latent.copy()
sc.pp.neighbors(
    view_hvg10000, use_rep="X_latent", n_neighbors=N_NEIGHBORS,
    metric="euclidean", transformer="sklearn", random_state=RANDOM_STATE,
)
sc.tl.umap(
    view_hvg10000, min_dist=UMAP_MIN_DIST, spread=UMAP_SPREAD,
    maxiter=UMAP_EPOCHS, random_state=RANDOM_STATE,
)
xy = np.asarray(view_hvg10000.obsm["X_umap"])
assert array_hash(xy) == fingerprints["X_umap_sha256"], "Original UMAP fingerprint mismatch."
draw_order = np.random.default_rng(RANDOM_STATE).permutation(len(meta))
lower, upper = xy.min(axis=0), xy.max(axis=0)
pad = np.maximum((upper - lower) * 0.08, 0.1)
UMAP_LIMITS = ((lower[0] - pad[0], upper[0] + pad[0]),
               (lower[1] - pad[1], upper[1] + pad[1]))


#5. =========== Shared layout and editable PDF export
def make_umap(title, colors):
    left, right, bottom, top = FIGURE_MARGINS
    width, height = AXIS_SIZE + left + right, AXIS_SIZE + bottom + top
    fig = plt.figure(figsize=(width, height), dpi=FIGURE_DPI, facecolor="white")
    ax = fig.add_axes([left / width, bottom / height, AXIS_SIZE / width, AXIS_SIZE / height])
    ax.scatter(xy[draw_order, 0], xy[draw_order, 1], c=colors[draw_order],
               s=POINT_SIZE, alpha=POINT_ALPHA, edgecolors="none", linewidths=0)
    ax.set(xlim=UMAP_LIMITS[0], ylim=UMAP_LIMITS[1])
    ax.set_box_aspect(1)
    ax.set_title(title, fontsize=TITLE_FONTSIZE, pad=TITLE_PAD, loc="center")
    ax.set_xlabel("UMAP1", fontsize=AXIS_FONTSIZE, labelpad=AXIS_LABEL_PAD)
    ax.set_ylabel("UMAP2", fontsize=AXIS_FONTSIZE, labelpad=AXIS_LABEL_PAD)
    ax.tick_params(labelsize=TICK_FONTSIZE, pad=TICK_PAD, length=TICK_LENGTH,
                   width=AXIS_LINEWIDTH, colors="black", direction="out")
    ax.spines[["top", "right"]].set_visible(False)
    for edge in ("left", "bottom"):
        ax.spines[edge].set(color="black", linewidth=AXIS_LINEWIDTH)
    ax.set_facecolor("white")
    ax.grid(False)
    return fig, ax


class ThreeGroupPdfRenderer(RendererPdf):
    """Write individual circle paths; clipping is applied once per object group."""
    draw_path_collection = RendererBase.draw_path_collection

    def draw_markers(self, gc, marker_path, marker_trans, path, trans, rgbFace=None):
        if len(marker_path.vertices) == 2:  # Tick marks are open stroke paths.
            rgbFace = None
        RendererBase.draw_markers(self, gc, marker_path, marker_trans, path, trans, rgbFace)

    def check_gc(self, gc, fillcolor=None):
        clean = self.new_gc()
        clean.copy_properties(gc)
        clean.set_clip_rectangle(None)
        clean.set_clip_path(None)
        super().check_gc(clean, fillcolor)


def export_umap_pdf(fig, ax, path):
    dpi = fig.dpi
    buffer = io.BytesIO()
    try:
        with mpl.rc_context({"pdf.fonttype": 42, "pdf.use14corefonts": False}):
            fig.set_dpi(72)
            fig.canvas.draw()
            width, height = fig.get_size_inches()
            pdf = PdfFile(buffer, metadata={"Title": ax.get_title()})
            pdf.newPage(width, height)
            renderer = ThreeGroupPdfRenderer(pdf, dpi, height, width)
            ticks = ax.xaxis._update_ticks() + ax.yaxis._update_ticks()
            points = list(ax.collections)
            lines = [ax.spines["left"], ax.spines["bottom"]] + list(ax.lines)
            lines += [tick.tick1line for tick in ticks]
            texts = [ax.title, ax.xaxis.label, ax.yaxis.label] + list(ax.texts)
            texts += [tick.label1 for tick in ticks]
            texts += [ax.xaxis.get_offset_text(), ax.yaxis.get_offset_text()]
            for name, artists in (("Scatter", points), ("Axes", lines), ("Text", texts)):
                pdf.write(f"% {name}\n".encode("ascii"))
                pdf.output(Op.gsave, Bbox.from_bounds(0, 0, width * 72, height * 72),
                           Op.rectangle, Op.clip, Op.endpath)
                renderer.gc = GraphicsContextPdf(pdf)
                for artist in artists:
                    if artist.get_visible():
                        artist.draw(renderer)
                renderer.finalize()
                pdf.output(Op.grestore)
            pdf.finalize()
            pdf.close()
    finally:
        fig.set_dpi(dpi)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_bytes(buffer.getvalue())


#6. =========== Cell annotation labels
point_colors = np.array([postbirth_stage_colors[group]
                         for group in meta["stage_celltype"].astype(str)])
fig, ax = make_umap("Postnatal: Cell annotation", point_colors)
anchors, texts = [], []
for i, group in enumerate(groups):
    points = xy[codes == i]
    anchor = points[np.argmin(cdist(points, points).sum(axis=1))]
    anchors.append(anchor)
    stage, annotation = group.replace("_", " ").split(" ", 1)
    texts.append(ax.text(*anchor, f"{stage}\n{annotation}",
                         ha="center", va="center", fontsize=LABEL_FONTSIZE,
                         color="black", zorder=4))

fig.canvas.draw()
renderer = fig.canvas.get_renderer()
axis_box = ax.get_window_extent(renderer)
pixels = ax.transData.transform(xy)
placed = []
for group, anchor, text in zip(groups, anchors, texts):
    center = ax.transData.transform(anchor)
    box = text.get_window_extent(renderer)
    w, h = box.width + 4, box.height + 3
    if group in LABEL_OFFSETS:
        offsets = [np.asarray(LABEL_OFFSETS[group]) * FIGURE_DPI / 72]
    else:
        offsets = [radius * FIGURE_DPI / 72 * np.array([np.cos(angle), np.sin(angle)])
                   for radius in LABEL_SEARCH_RADII
                   for angle in ([0] if radius == 0 else np.linspace(0, 2 * np.pi, 16, endpoint=False))]
    choices = []
    for offset in offsets:
        pos = center + offset
        candidate = Bbox.from_bounds(pos[0] - w / 2, pos[1] - h / 2, w, h)
        if not (axis_box.contains(*candidate.p0) and axis_box.contains(*candidate.p1)):
            continue
        if any(candidate.overlaps(other) for other in placed):
            continue
        covered = np.count_nonzero((pixels[:, 0] >= candidate.x0) & (pixels[:, 0] <= candidate.x1)
                                   & (pixels[:, 1] >= candidate.y0) & (pixels[:, 1] <= candidate.y1))
        choices.append((20 * covered + np.linalg.norm(offset), pos, candidate))
    assert choices, f"Adjust LABEL_OFFSETS for {group}."
    _, pos, candidate = min(choices, key=lambda item: item[0])
    text.set_position(ax.transData.inverted().transform(pos))
    placed.append(candidate)

fig.canvas.draw()
boxes = [text.get_window_extent(fig.canvas.get_renderer()) for text in texts]
assert not any(boxes[i].overlaps(boxes[j]) for i in range(len(boxes)) for j in range(i + 1, len(boxes)))
np.testing.assert_allclose(ax.get_window_extent().size / fig.dpi, [AXIS_SIZE, AXIS_SIZE])
# Set the output file path.
export_umap_pdf(fig, ax, ROOT)
plt.show()
plt.close(fig)


In [ ]:
#7. =========== Stage distribution on the same UMAP
STAGE_LEGEND_X = 1.04
STAGE_LEGEND_Y = 0.54
STAGE_LEGEND_GAP = 0.045

stage_xy = np.asarray(view_hvg10000.obsm["X_umap"])
np.testing.assert_array_equal(stage_xy, xy)
stage_values = view_hvg10000.obs["stage_analysis"].astype(str).to_numpy()
stage_point_colors = np.array([STAGE_COLORS[stage] for stage in stage_values])
fig, ax = make_umap("Postnatal: Stage distribution", stage_point_colors)

for i, stage in enumerate(STAGE_ORDER):
    y = STAGE_LEGEND_Y - i * STAGE_LEGEND_GAP
    ax.scatter([STAGE_LEGEND_X], [y], transform=ax.transAxes,
               s=LEGEND_MARKER_SIZE ** 2, c=[STAGE_COLORS[stage]],
               alpha=POINT_ALPHA, edgecolors="none", linewidths=0, clip_on=False)
    count = np.count_nonzero(stage_values == stage)
    ax.text(STAGE_LEGEND_X + 0.025, y, f"{stage} ({count})",
            transform=ax.transAxes, fontsize=LEGEND_FONTSIZE,
            color="black", ha="left", va="center", clip_on=False)

fig.canvas.draw()
np.testing.assert_allclose(ax.get_window_extent().size / fig.dpi, [AXIS_SIZE, AXIS_SIZE])
np.testing.assert_array_equal(ax.get_xlim(), UMAP_LIMITS[0])
np.testing.assert_array_equal(ax.get_ylim(), UMAP_LIMITS[1])
# Set the output file path.
export_umap_pdf(fig, ax, ROOT)
plt.show()
plt.close(fig)


In [ ]:
#8. =========== Cell annotation with a complete legend
ANNOTATION_LEGEND_X = 1.04
ANNOTATION_LEGEND_TOP = 0.695
ANNOTATION_LEGEND_GAP = 0.035

fig, ax = make_umap("Postnatal: Cell annotation", point_colors)
annotation_counts = meta["stage_celltype"].value_counts(sort=False)
legend_groups = [group for group in groups if annotation_counts[group] > 0]
legend_texts = []
for i, group in enumerate(legend_groups):
    y = ANNOTATION_LEGEND_TOP - i * ANNOTATION_LEGEND_GAP
    ax.scatter([ANNOTATION_LEGEND_X], [y], transform=ax.transAxes,
               s=LEGEND_MARKER_SIZE ** 2, c=[postbirth_stage_colors[group]],
               alpha=POINT_ALPHA, edgecolors="none", linewidths=0, clip_on=False)
    legend_texts.append(ax.text(
        ANNOTATION_LEGEND_X + 0.025, y, f"{group} ({annotation_counts[group]})",
        transform=ax.transAxes, fontsize=LEGEND_FONTSIZE,
        color="black", ha="left", va="center", clip_on=False,
    ))

fig.canvas.draw()
legend_boxes = [text.get_window_extent(fig.canvas.get_renderer()) for text in legend_texts]
assert all(fig.bbox.contains(*box.p0) and fig.bbox.contains(*box.p1) for box in legend_boxes)
assert not any(legend_boxes[i].overlaps(legend_boxes[j])
               for i in range(len(legend_boxes)) for j in range(i + 1, len(legend_boxes)))
np.testing.assert_allclose(ax.get_window_extent().size / fig.dpi, [AXIS_SIZE, AXIS_SIZE])
np.testing.assert_array_equal(ax.collections[0].get_offsets(), xy[draw_order])
# Set the output file path.
export_umap_pdf(fig, ax, ROOT)
plt.show()
plt.close(fig)


In [ ]:
#9. =========== Stage distribution without a legend
fig, ax = make_umap("Postnatal: Stage distribution", stage_point_colors)

fig.canvas.draw()
np.testing.assert_allclose(ax.get_window_extent().size / fig.dpi, [AXIS_SIZE, AXIS_SIZE])
np.testing.assert_array_equal(ax.collections[0].get_offsets(), xy[draw_order])
np.testing.assert_array_equal(ax.get_xlim(), UMAP_LIMITS[0])
np.testing.assert_array_equal(ax.get_ylim(), UMAP_LIMITS[1])
# Set the output file path.
export_umap_pdf(fig, ax, ROOT)
plt.show()
plt.close(fig)
